# 03 — Graph Machine Learning
**GCN + GraphSAGE Node Classification**

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')
%matplotlib inline

print(f'PyTorch: {torch.__version__}')
print(f'Device : {"cuda" if torch.cuda.is_available() else "cpu"}')

## 1. Load Data + Build Graph

In [ ]:
from src.graph_builder import load_processed, build_graph
from src.centrality import build_centrality_table
from src.communities import detect_louvain

nodes, edges = load_processed()
G            = build_graph(nodes, edges)
cent_df      = build_centrality_table(G)
partition    = detect_louvain(G)

print(f'Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}')
print(f'Communities: {len(set(partition.values()))}')

## 2. Prepare PyG Data

In [ ]:
from src.graph_ml import prepare_graph_data

data, label_encoder, node_ids = prepare_graph_data(
    G, cent_df, partition,
    test_size=0.25, min_community_size=3,
)

print(f'Feature matrix : {data.x.shape}')
print(f'Labels         : {data.y.shape}')
print(f'Train nodes    : {data.train_mask.sum().item()}')
print(f'Test nodes     : {data.test_mask.sum().item()}')
print(f'Classes        : {len(label_encoder.classes_)}')

## 3. Train GCN

In [ ]:
from src.graph_ml import GCN, train_model, evaluate_model

in_ch  = data.x.shape[1]
n_cls  = int(data.y.max().item()) + 1

gcn = GCN(in_channels=in_ch, hidden_channels=64,
          out_channels=n_cls, dropout=0.4)

gcn, gcn_train_loss, gcn_test_loss = train_model(
    gcn, data, model_name='GCN',
    epochs=300, lr=0.005, patience=35
)

## 4. Train GraphSAGE

In [ ]:
from src.graph_ml import GraphSAGE

sage = GraphSAGE(in_channels=in_ch, hidden_channels=64,
                 out_channels=n_cls, dropout=0.4)

sage, sage_train_loss, sage_test_loss = train_model(
    sage, data, model_name='GraphSAGE',
    epochs=300, lr=0.005, patience=35
)

## 5. Evaluate Both Models

In [ ]:
gcn_res  = evaluate_model(gcn,  data, label_encoder, 'GCN')
sage_res = evaluate_model(sage, data, label_encoder, 'GraphSAGE')

print(f"\nGCN  Accuracy: {gcn_res['accuracy']:.4f}  "
      f"F1: {gcn_res['f1_weighted']:.4f}")
print(f"SAGE Accuracy: {sage_res['accuracy']:.4f}  "
      f"F1: {sage_res['f1_weighted']:.4f}")

## 6. Training Curves

In [ ]:
from src.graph_ml import plot_training_curves
plot_training_curves(
    gcn_train_loss, gcn_test_loss,
    sage_train_loss, sage_test_loss
)
from IPython.display import Image
Image('outputs/figures/training_curves.png')

## 7. Confusion Matrices

In [ ]:
from src.graph_ml import plot_confusion_matrix
plot_confusion_matrix(gcn_res,  label_encoder, 'GCN')
plot_confusion_matrix(sage_res, label_encoder, 'GraphSAGE')

from IPython.display import Image, display
display(Image('outputs/figures/confusion_matrix_gcn.png'))
display(Image('outputs/figures/confusion_matrix_graphsage.png'))

## 8. Save Models

In [ ]:
from src.graph_ml import save_model
save_model(gcn,  'gcn')
save_model(sage, 'graphsage')
print('Models saved to outputs/models/ ✅')